In [18]:
import os
import re
import json
import numpy as np
import torch
import PyPDF2
import faiss
import random
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

# ==========================================
# 1. KONFIGURASI NAMA FILE
# ==========================================
PDF_PATH   = "ALKITAB PDF-3-1162.pdf"  # Pastikan file ini ada
FILE_FAISS = "bible_index.faiss"       # File Struktur Index
FILE_EMBED = "embeddings.npy"          # File Matriks Angka
FILE_META  = "verses_ref.json"         # File Teks Ayat

In [19]:
def extract_text_from_pdf(pdf_path):
    text = ""
    if not os.path.exists(pdf_path):
        print(f"❌ File {pdf_path} tidak ditemukan!")
        return None
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in tqdm(reader.pages, desc="📖 Membaca PDF"):
            t = page.extract_text()
            if t: text += "\n" + t
    return text

def parse_bible_complex(text):
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    data_items = []
    current_book = None
    current_pericope_title = "Umum"
    buffer_text = ""
    buffer_ref = ""
    
    regex_pericope_ref = re.compile(r'^\(\s*(.+?)\s+(\d+:\d+(?:\s?[-–]\s?(?:\d+:\d+|\d+))?)\s*\)$')
    regex_verse_start = re.compile(r'^(\d+):(\d+)\s+(.*)')
    regex_chapter_only = re.compile(r'^\d+$')

    for i, line in enumerate(lines):
        # Deteksi Perikop
        match_peri = regex_pericope_ref.match(line)
        if match_peri:
            if buffer_ref and buffer_text:
                data_items.append({"type": "ayat", "ref": buffer_ref, "text": buffer_text.strip(), "parent": current_pericope_title})
                buffer_text = ""; buffer_ref = ""
            if i > 0:
                judul_potensial = lines[i-1]
                if not regex_chapter_only.match(judul_potensial) and not regex_verse_start.match(judul_potensial):
                    current_pericope_title = judul_potensial
                    kitab = match_peri.group(1).strip()
                    ref_range = match_peri.group(2).strip()
                    data_items.append({"type": "perikop", "ref": f"{kitab} {ref_range}", "text": current_pericope_title, "parent": "-"})
                    current_book = kitab
            continue

        # Deteksi Ayat
        match_verse = regex_verse_start.match(line)
        if match_verse:
            if buffer_ref and buffer_text:
                data_items.append({"type": "ayat", "ref": buffer_ref, "text": buffer_text.strip(), "parent": current_pericope_title})
            pasal, ayat, isi = match_verse.groups()
            book_name = current_book if current_book else "Kitab"
            buffer_ref = f"{book_name} {pasal}:{ayat}"
            buffer_text = isi
            continue
        
        if regex_chapter_only.match(line): continue
        
        # Handle Text Wrapping
        is_next_line_ref = False
        if i + 1 < len(lines):
            if regex_pericope_ref.match(lines[i+1]): is_next_line_ref = True
        if buffer_ref and not is_next_line_ref:
            buffer_text += " " + line

    if buffer_ref and buffer_text:
        data_items.append({"type": "ayat", "ref": buffer_ref, "text": buffer_text.strip(), "parent": current_pericope_title})
    return data_items

In [20]:
def save_system_data(index, embeddings, dataset):
    print("\n💾 MENYIMPAN SYSTEM KE FILE...")
    faiss.write_index(index, FILE_FAISS)
    np.save(FILE_EMBED, embeddings)
    with open(FILE_META, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)
    print("✅ Berhasil menyimpan 3 file utama (FAISS, NPY, JSON).")

def load_or_create_system():
    # Load Model AI (Wajib)
    print("⚙️ Memuat Model AI (SentenceTransformer & CrossEncoder)...")
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    # CEK APAKAH FILE SUDAH ADA?
    if os.path.exists(FILE_FAISS) and os.path.exists(FILE_META) and os.path.exists(FILE_EMBED):
        print("\n📂 FILE DITEMUKAN! Memuat data dari file lokal...")
        # Load FAISS
        index = faiss.read_index(FILE_FAISS)
        # Load JSON
        with open(FILE_META, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
        # Load NPY (Opsional, buat jaga-jaga)
        embeddings = np.load(FILE_EMBED)
        
        print(f"✅ Data Terload: {len(dataset)} items.")
        return model, cross_encoder, index, dataset
    
    else:
        print("\n⚠️ FILE TIDAK DITEMUKAN. Memulai proses Ekstraksi PDF (Hanya sekali)...")
        raw_text = extract_text_from_pdf(PDF_PATH)
        if not raw_text: return None, None, None, None
        
        dataset = parse_bible_complex(raw_text)
        
        print("🧠 Membuat Embedding...")
        texts = [item['text'] for item in dataset]
        embeddings = model.encode(texts, show_progress_bar=True, batch_size=64).astype('float32')
        
        print("🏗️ Membangun Index FAISS...")
        dim = embeddings.shape[1]
        index = faiss.IndexFlatL2(dim)
        index.add(embeddings)
        
        # SIMPAN HASIL AGAR BESOK TIDAK PERLU ULANG
        save_system_data(index, embeddings, dataset)
        
        return model, cross_encoder, index, dataset

In [21]:
def search_and_yapping(query, model, index, dataset, cross_encoder):
    print(f"\n🔎 Sedang mencari: '{query}'...")
    
    # 1. Retrieve (Ambil 50 kandidat)
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, 50)
    
    candidates = []
    candidate_indices = []
    for idx in indices[0]:
        if idx < len(dataset):
            candidates.append([query, dataset[idx]['text']])
            candidate_indices.append(idx)
            
    if not candidates:
        print("❌ Tidak ada hasil.")
        return

    # 2. Re-Rank (AI Menilai ulang)
    scores = cross_encoder.predict(candidates)
    ranked_results = sorted(list(zip(candidate_indices, scores)), key=lambda x: x[1], reverse=True)
    
    # 3. Pisahkan Perikop & Ayat
    results_perikop = []
    results_ayat = []
    seen_refs = set()

    for idx, score in ranked_results:
        item = dataset[idx]
        if item['ref'] in seen_refs: continue
        seen_refs.add(item['ref'])
        
        if item['type'] == 'perikop':
            results_perikop.append(item)
        else:
            results_ayat.append(item)

    # ================= OUTPUT =================
    
    print("\n📂 5 PERIKOP TERKAIT:")
    print("-" * 30)
    for p in results_perikop[:5]:
        print(f"🔹 {p['text']} ({p['ref']})")

    print("\n📖 5 AYAT TERATAS:")
    print("-" * 30)
    final_ayat_list = results_ayat[:5]
    for i, res in enumerate(final_ayat_list, 1):
        print(f"{i}. [{res['ref']}] {res['text'][:100]}...")

    # ================= YAPPING SECTION =================
    if final_ayat_list:
        top_item = final_ayat_list[0] # Ambil Juara 1
        
        # Pilihan kata-kata penutup acak
        closings = [
            "Semoga ini menjadi kekuatan baru bagi kita. Amin! 🙏",
            "Biarlah firman ini menerangi langkah kita hari ini. ✨",
            "Tuhan Yesus memberkati kita semua melalui firman-Nya. ❤️"
            "Mudah-mudahan kita bisa dikuatkan dari sana. Tuhan memberkati. 🙏",
            "Kiranya ayat ini menjadi penguat hati kita hari ini. Amin. 🙌",
            "Semoga ini menjadi pengingat dan rhema yang indah untuk hari ini. ✨",
            "Biarlah firman ini menjadi pelita bagi langkah kaki kita hari ini. 🕯",
            "Kiranya damai sejahtera menyertai kita melalui kebenaran ini. 🕊",
            "Tuhan Yesus memberkati kita semua lewat sapaan firman-Nya ini. ❤"
        ]
        chosen_closing = random.choice(closings)
        
        # Tampilkan Yapping
        print("\n" + "="*60)
        print("✨AYAT TERATAS HARI INI ✨")
        print("="*60)
        print(f"Topik    : {query}")
        print(f"Referensi: {top_item['ref']}")
        print(f"Konteks  : {top_item['parent']}")
        print("-" * 60)
        print(f"\"{top_item['text']}\"")
        print("-" * 60)
        print(f"💡 Pesan Singkat:\nSobat, lihatlah bagaimana ayat {top_item['ref']} ini\n"
              f"menjawab kerinduan hati kita tentang '{query}'.\n"
              f"{chosen_closing}")
        print("="*60)
    else:
        print("\n❌ Belum ada ayat yang pas untuk di-yapping-kan.")

In [44]:
# A. Load Sistem (Otomatis cek file atau buat baru)
model, cross_encoder, index, dataset = load_or_create_system()

# B. Masukkan Kata Kunci Disini
KEYWORD = "Kasih Tuhan"  

# C. Jalankan Pencarian
if dataset:
    search_and_yapping(KEYWORD, model, index, dataset, cross_encoder)

⚙️ Memuat Model AI (SentenceTransformer & CrossEncoder)...

📂 FILE DITEMUKAN! Memuat data dari file lokal...
✅ Data Terload: 33414 items.

🔎 Sedang mencari: 'Kasih Tuhan'...

📊 HASIL PENCARIAN DENGAN DUAL METRICS

📂 5 PERIKOP TERKAIT (Diurutkan by Cross-Encoder Score):
------------------------------------------------------------------------------------------
No   Perikop                        L2 Distance     CE Score    
------------------------------------------------------------------------------------------
1    Murka TUHAN                    5.3082 █████ 5.2202 ███████████████
2    Kedatangan Tuhan               4.5596 ████  4.3413 █████████████
3    Allah adalah kasih             4.5397 ████  2.7995 ████████
4    Kebesaran TUHAN                6.9833 ██████ 2.6823 ████████
5    Kasih kepada Allah adalah pe   7.5101 ███████ 2.4686 ███████ 

📖 5 AYAT TERATAS (Diurutkan by Cross-Encoder Score):
-----------------------------------------------------------------------------------------